# 04: Train Reranker on Colab GPU

This notebook is the GPU training stage for the Cross-Encoder reranker. S3/DVC is the source of truth; Google Drive can be used only as a local Colab cache.

## Runtime

Use `Runtime -> Change runtime type -> T4/A100` before running. The notebook pulls prepared artifacts from S3, trains the reranker, stores the checkpoint under `artifacts/experiments/<RUN_ID>/models/reranker`, then pushes artifacts back to S3 through DVC.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU in Colab: Runtime -> Change runtime type -> T4/A100."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
REPO_OWNER = "terrylimax"
REPO_NAME = "medical-rag-reranker"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
BRANCH = os.getenv("PROJECT_BRANCH", "feature/remote-storage-vllm")
PROJECT_DIR = Path("/content") / REPO_NAME

RUN_ID = os.getenv("EXPERIMENT_RUN_ID", "medquad_full_remote")
OUTPUT_DIR = Path("artifacts") / "experiments" / RUN_ID
RERANKER_DIR = OUTPUT_DIR / "models" / "reranker"

print("RUN_ID:", RUN_ID)
print("Output dir:", OUTPUT_DIR)

In [ ]:
def sh(*args, cwd=None):
    args = [str(arg) for arg in args]
    print("+", " ".join(args))
    subprocess.run(args, cwd=None if cwd is None else str(cwd), check=True)


if not PROJECT_DIR.exists():
    sh("git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=PROJECT_DIR)
    sh("git", "checkout", BRANCH, cwd=PROJECT_DIR)
    sh("git", "pull", "--ff-only", cwd=PROJECT_DIR)

os.chdir(PROJECT_DIR)
sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".")

## Secrets and remote storage

Set these in Colab secrets or environment before running: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_REGION`, `ARTIFACT_DVC_REMOTE`, and optionally `ARTIFACT_REMOTE_URI`. API keys are not written to notebook outputs.

In [ ]:
try:
    from google.colab import userdata
except Exception:
    userdata = None


def set_secret(name, default=None):
    if os.getenv(name):
        return
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            return
    if default is not None:
        os.environ[name] = default


set_secret("AWS_ACCESS_KEY_ID")
set_secret("AWS_SECRET_ACCESS_KEY")
set_secret("AWS_REGION", "eu-north-1")
set_secret("ARTIFACT_DVC_REMOTE", "artifact_s3")
set_secret("ARTIFACT_REMOTE_URI")

print("DVC remote:", os.getenv("ARTIFACT_DVC_REMOTE"))
print("AWS region:", os.getenv("AWS_REGION"))
print("Artifact remote URI set:", bool(os.getenv("ARTIFACT_REMOTE_URI")))

In [ ]:
sh(sys.executable, "-m", "medical_rag_reranker.commands", "artifact_pull")

## Train Cross-Encoder reranker

This uses the full dataloaders by setting `limit_train_batches=1.0` and `limit_val_batches=1.0`. Tune batch size and epochs depending on the GPU.

In [ ]:
overrides = [
    f"train.max_epochs={int(os.getenv('RERANKER_EPOCHS', '2'))}",
    f"train.batch_size={int(os.getenv('RERANKER_BATCH_SIZE', '8'))}",
    "train.limit_train_batches=1.0",
    "train.limit_val_batches=1.0",
    "train.accelerator=gpu",
    "train.devices=1",
    f"logging.run_name=reranker_{RUN_ID}",
]

sh(
    sys.executable,
    "-m",
    "medical_rag_reranker.commands",
    "train",
    "--overrides",
    __import__("json").dumps(overrides),
)

## Persist artifacts

Copy or move the resulting checkpoint into the experiment artifact directory if the training command writes it elsewhere, then push the experiment folder and DVC metadata to S3.

In [ ]:
RERANKER_DIR.mkdir(parents=True, exist_ok=True)
print("Expected reranker artifact dir:", RERANKER_DIR.resolve())

# Push all current runtime artifacts plus experiment outputs.
sh(
    sys.executable,
    "-m",
    "medical_rag_reranker.commands",
    "artifact_push",
    "--include",
    f"artifacts/experiments/{RUN_ID}/**/*",
)